In [3]:
import nltk
# nltk.download('averaged_perceptron_tagger')
import traceback
import os
import chardet
import magic
import pandas as pd
import polars as pl

from langchain_docling import DoclingLoader
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_community.document_loaders.pdf import ZeroxPDFLoader
from langchain_teddynote.document_loaders import HWPLoader
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyMuPDFLoader,
    UnstructuredPowerPointLoader,
    Docx2txtLoader,
    DataFrameLoader,
    CSVLoader,
    UnstructuredCSVLoader,
    PolarsDataFrameLoader,
    UnstructuredExcelLoader,
    UnstructuredTSVLoader,
    UnstructuredWordDocumentLoader,
    PyMuPDFLoader,
    PDFMinerLoader,
    PDFPlumberLoader,
    PyPDFLoader,
    PyPDFDirectoryLoader,
    PyPDFium2Loader,
    UnstructuredPDFLoader,
    UnstructuredTSVLoader,
    UnstructuredOrgModeLoader,
    UnstructuredMarkdownLoader,
    DataFrameLoader,
    PolarsDataFrameLoader,
    UnstructuredOrgModeLoader
)

EXT_LOADER_ROUTER = {
    ".txt": TextLoader,
    ".md": TextLoader,
    ".org": UnstructuredOrgModeLoader,
    ".docx": Docx2txtLoader,
    ".doc": UnstructuredExcelLoader,
    ".pdf": PyMuPDFLoader,
    ".xlsx": UnstructuredExcelLoader,
    ".xls": DoclingLoader,
    ".csv": CSVLoader,
    ".tsv": UnstructuredTSVLoader,
    ".pptx": UnstructuredPowerPointLoader,
    ".ppt": UnstructuredPowerPointLoader,
    ".hwp": HWPLoader
}

def get_loader(path):
    ext = os.path.splitext(path)[1].lower()
    loader_cls = EXT_LOADER_ROUTER.get(ext, UnstructuredLoader)

    if ext == ".txt":
        return loader_cls(path, encoding="utf-8")
    elif ext == ".csv":
        return loader_cls(path, autodetect_encoding=True)
    else:
        return loader_cls(path)

# 폴더 내 모든 파일 경로 수집
def get_file_paths(folder_path):
    file_paths = []
    for file in os.listdir(folder_path):
        full_path = os.path.join(folder_path, file)
        if os.path.isfile(full_path):
            file_paths.append(full_path)
    return file_paths

def connect_loader(folder_path, extensions=None):
    docs = []
    failed_files = []
    file_paths = get_file_paths(folder_path)
    for file_path in file_paths:
        ext = os.path.splitext(file_path)[1].lower()
        if extensions is not None and ext not in extensions:
            continue
        loader = get_loader(file_path)
        try:
            loaded = loader.load()
            docs.append(loaded)
        except Exception as e:
            print(f"[ERROR] 파일 로드 실패: {file_path} ({type(e).__name__}) - {e}")
            failed_files.append(file_path)
            continue
    return docs, failed_files


In [ ]:
docs,failed_files = connect_loader('../test-files/documents/doc', extensions={".doc"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/presentations/ppt', extensions={".ppt"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/spreadsheets/csv', extensions={".csv"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/spreadsheets/tsv', extensions={".tsv"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/spreadsheets/xls', extensions={".xls"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/texts/txt', extensions={".txt"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/texts/rtf', extensions={".rtf"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/pdfs', extensions={".pdf"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/texts/org', extensions={".org"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
docs,failed_files = connect_loader('../test-files/texts/md', extensions={".md"})
[
    f"전체 문서 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}"
]

In [ ]:
contents = [doc.page_content for doc in docs]
contents

가장 적합한 Loader 찾기

-   txt
    -   DoclingLoader
        -   지금 txt확장자 encoding 파라미터 넘기고있는데 DoclingLoader는 확장자 파라미터 안받아서 분기처리 추가해야 할듯
    -   TextLoader
        -   성공: 98, 에러: 0
        -   0.3s
-   md
    -   TextLoader
        -   성공: 9, 에러: 0
        -   0.2s
    -   DoclingLoader
        -   성공: 38730, 실패: 0
        -   11m 16.8s
-   rtf
    -   UnstructuredLoader
        -   성공: 217, 에러: 0
        -   17.5s
-   org
    -   UnstructuredOrgModeLoader
        -   성공: 8, 에러:0
        -   2s
-   docx
    -   Docx2txtLoader
        -   linear에 없음
        -   성공: 127, 에러: 16
        -   4.3s
    -   DoclingLoader
        -   성공: 8875, 에러: 19
        -   3m 2.5s
    -   UnstructuredWordDocumentLoader
        -   성공: 125, 에러: 18
        -   2m 13s
-   doc
    -   Docx2txtLoader
        -   성공: 0, 에러: 158
        -   0.3s
    -   DoclingLoader
        -   성공: 0, 에러: 158
        -   2m 26.5s
    -   UnstructuredWordDocumentLoader
        -   성공: 150, 에러: 8
        -   16m 1.8s
-   pdf
    -   PyMuPDFLoader
        -   성공: 66, 에러: 0
        -   0.1s
        -   pdf 1개당 document 1개가 아님
    -   PDFMinerLoader
        -   성공: 8, 에러: 0
        -   2.2s
    -   PDFPlumberLoader
        -   성공: 66, 에러: 0
        -   4.0s
    -   PyPDFLoader
        -   성공: 66, 에러: 0
        -   1.3s
    -   PyPDFDirectoryLoader
        -   디렉토리 로더라 사용법이 달라서 일단 테스트안함
    -   PyPDFium2Loader
        -   성공: 66, 에러: 0
        -   0.4s
    -   UnstructuredPDFLoader
        -   성공:0 , 에러: 8
        -   (AttributeError) - module 'ml_dtypes' has no attribute 'float4_e2m1fn' 에러 발생
        -   11.9s
-   xlsx
    -   DoclingLoader
        -   jupyter 커널 팅김
            -   [error] Disposing session as kernel process died ExitCode: undefined, Reason:
    -   UnstructuredExcelLoader
        -   jupyter 커널 팅김
            -   [error] Disposing session as kernel process died ExitCode: undefined, Reason:
    -   DataFrameLoader
    -   PolarsDataFrameLoader
-   xls
    -   DoclingLoader
        -   성공: 0, 에러: 412
        -   7m 19.7s
    -   DataFrameLoader
-   csv
    -   TextLoader
        -   성공: 148, 에러: 0
        -   1.1s
    -   CSVLoader
        -   성공: 450570, 에러: 0
        -   5.3s
        -   우리는 파일 1개(document) 단위로 청킹하니까 어차피 CSVLoader를 써도 1개로 합칠거기때문에 TextLoader를 쓴다면 더 빠름
    -   DataFrameLoader
    -   CSV 파일을 직접 읽어서 하나의 문자열로 합친 뒤, Document 객체로 생성하는 커스텀 로더
-   tsv
    -   UnstructuredTSVLoader
        -   성공: 10, 에러: 1
        -   0.3s
    -   DataFrameLoader
-   pptx
    -   DoclingLoader
        -   성공: 465, 에러: 11
        -   1m 17.6s
    -   UnstructuredPowerPointLoader
        -   성공: 79, 에러: 9
        -   3s
-   ppt
    -   DoclingLoader
        -   연결: 0, 에러: 134
        -   2m 17.6s
    -   UnstructuredPowerPointLoader
        -   연결: 113, 에러: 21
        -   43m 27.6s
        -   pdf로 변환하는거 생각하기
-   hwp
    -   HWPLoader
        -   성공: 16, 에러: 0
        -   0.1s
